In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset

import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.amp import GradScaler, autocast
import matplotlib.pyplot as plt


In [2]:
def get_dataloaders(dataset_name, batch_size, pin_memory):
    # ResNet expects 3 channels; we duplicate the grayscale channel
    transform = transforms.Compose([
        transforms.Resize((32, 32)), # Resize slightly for ResNet stability
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == 'MNIST':
        full_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else: # FashionMNIST
        full_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # Merge to re-split according to 70-10-20
    dataset = ConcatDataset([full_data, test_data])
    total_len = len(dataset)
    train_len = int(0.7 * total_len)
    val_len = int(0.1 * total_len)
    test_len = total_len - train_len - val_len

    train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

    return train_loader, val_loader, test_loader

In [6]:
!pip install thop

In [7]:
import time
from thop import profile  # Install via: pip install thop

def get_model(model_name, num_classes=10):
    # Re-defining locally to ensure clean state
    if model_name == 'ResNet-18':
        model = models.resnet18(weights=None)
    elif model_name == 'ResNet-50':
        model = models.resnet50(weights=None)
    # Note: ResNet-32 is not in standard torchvision; skipped to keep code minimal.

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def run_hardware_experiment(device_type, model_name, batch_size, opt_name, lr, epochs=1):
    device = torch.device(device_type)
    print(f"Running: {device_type} | {model_name} | {opt_name} | LR:{lr}")

    # 1. Data Setup (FashionMNIST)
    train_loader, _, test_loader = get_dataloaders('FashionMNIST', batch_size, pin_memory=False)

    # 2. Model Setup
    model = get_model(model_name).to(device)

    # 3. Calculate FLOPs (using a dummy input on the correct device)
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)

    # 4. Optimizer
    criterion = nn.CrossEntropyLoss()
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else: # Adam
        optimizer = optim.Adam(model.parameters(), lr=lr)

    # 5. Training with Timing
    model.train()

    # Synchronization for accurate GPU timing
    if device_type == 'cuda': torch.cuda.synchronize()
    start_time = time.time()

    for epoch in range(epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    if device_type == 'cuda': torch.cuda.synchronize()
    end_time = time.time()

    total_train_time_ms = (end_time - start_time) * 1000  # Convert seconds to ms

    # 6. Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy, total_train_time_ms, flops


In [9]:
# --- Run Experiments for Table 2 ---
hardware_configs = [
    ('cuda', 16, 'SGD', 0.001),
    ('cuda', 16, 'Adam', 0.001),
    ('cpu', 16, 'SGD', 0.001),
    ('cpu', 16, 'Adam', 0.001)
]

print(f"{'Device':<6} | {'BS':<3} | {'Opt':<5} | {'Model':<9} | {'Acc (%)':<8} | {'Time (ms)':<10} | {'FLOPs'}")
print("-" * 75)

for dev, bs, opt, lr in hardware_configs:
    if dev == 'cuda' and not torch.cuda.is_available():
        print(f"Skipping {dev} (not available)")
        continue

    for m_name in ['ResNet-18', 'ResNet-50']:
        # Run for small epochs (e.g., 1) just to get timing/flops as per requirement
        acc, t_time, flops = run_hardware_experiment(dev, m_name, bs, opt, lr, epochs=1)

        print(f"{dev:<6} | {bs:<3} | {opt:<5} | {m_name:<9} | {acc:.2f}%   | {t_time:.0f} ms | {flops:.0f}")

Device | BS  | Opt   | Model     | Acc (%)  | Time (ms)  | FLOPs
---------------------------------------------------------------------------
Running: cuda | ResNet-18 | SGD | LR:0.001
cuda   | 16  | SGD   | ResNet-18 | 86.99%   | 43956 ms | 37220352
Running: cuda | ResNet-50 | SGD | LR:0.001
cuda   | 16  | SGD   | ResNet-50 | 81.93%   | 75700 ms | 84342784
Running: cuda | ResNet-18 | Adam | LR:0.001
cuda   | 16  | Adam  | ResNet-18 | 85.13%   | 44321 ms | 37220352
Running: cuda | ResNet-50 | Adam | LR:0.001
cuda   | 16  | Adam  | ResNet-50 | 80.60%   | 87726 ms | 84342784
Running: cpu | ResNet-18 | SGD | LR:0.001
cpu    | 16  | SGD   | ResNet-18 | 85.94%   | 578479 ms | 37220352
Running: cpu | ResNet-50 | SGD | LR:0.001
cpu    | 16  | SGD   | ResNet-50 | 83.16%   | 1381111 ms | 84342784
Running: cpu | ResNet-18 | Adam | LR:0.001
cpu    | 16  | Adam  | ResNet-18 | 87.40%   | 816487 ms | 37220352
Running: cpu | ResNet-50 | Adam | LR:0.001
cpu    | 16  | Adam  | ResNet-50 | 82.98%   | 175